Task 1

In [ ]:
# Loading the data 
from pathlib import Path

import kagglehub
import pandas as pd

COMPETITION_NAME = "50-007-machine-learning-may-2026"

competition_path = Path(kagglehub.competition_download(COMPETITION_NAME))

train_features_df = pd.read_csv(competition_path / "train_features.csv")

test_features_df = pd.read_csv(competition_path / "test_features.csv")

submission_df = pd.read_csv(competition_path / "sample_submission.csv")

In [ ]:
# import packages
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Test dataset features
X_test_submission = test_features_df.iloc[:, 1:]

# Original train dataset
X = train_features_df.iloc[:, 2:]
y = train_features_df.iloc[:, 1]

# Using the train test split function (75% for training, 25% for validating)
X_train, X_validate, y_train, y_validate = train_test_split(X, y, random_state=10, test_size=0.25, shuffle=True)

In [ ]:
def sigmoid(z):
    return 1.0/(1 + np.exp(-z))

def fit_normalizer(X):
    """Return the mean/std of the TRAINING data.

    These must be reused at prediction time. Recomputing mean/std from the
    set being predicted would rescale it differently from the data the
    weights were learned on, so the model would be applying its coefficients
    to a different feature space.
    """
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1, std)   # guard constant features
    return mean, std


def apply_normalizer(X, mean, std):
    return (X - mean) / std

def loss(y, y_hat):
    epsilon = 1e-15
    y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # Ensures that NaN is not returned for log(0)
    loss = -np.mean(y * (np.log(y_hat)) + (1 - y) * np.log(1-y_hat))
    return loss

def gradients(X, y, y_hat): 
    m = X.shape[0] # number of training examples

    # Gradient of loss w.r.t weights.
    dw = (1 / m) * np.dot(X.T, (y_hat - y))

    # Gradient of loss w.r.t bias.
    db = (1 / m) * np.sum((y_hat - y)) 

    return dw, db

def train(X, y, epochs, lr):
    # m-> number of training examples
    # n-> number of features 
    m, n = X.shape

    # Initializing weights and bias to zeros.
    w = np.zeros((n,1))
    b = 0

    # Reshaping y.
    y = np.asarray(y, dtype=float).reshape(-1, 1)

    # Normalizing the inputs; keep the stats so predict() can reuse them.
    mu, sigma = fit_normalizer(X)
    x = apply_normalizer(X, mu, sigma)

    # Empty list to store losses.
    losses = []

    previous_loss = float("inf")

    # Training loop.
    for epoch in range(epochs):

        # Calculating hypothesis/prediction.
        y_hat = sigmoid(np.dot(x, w) + b)

        # Getting the gradients of loss w.r.t parameters.
        dw, db = gradients(x, y, y_hat)

        # Updating the parameters.
        w -= lr * dw
        b -= lr * db

        # Calculating loss and appending it in the list.
        current_loss = loss(y, sigmoid(x @ w + b))
        losses.append(current_loss)

        if abs(previous_loss - current_loss) < 1e-6: # Stop after convergence
            print(f"Converged at epoch {epoch}")
            break

        previous_loss = current_loss

    # returning weights, bias, losses AND the normalisation stats
    return w, b, losses, mu, sigma

def predict_proba(X, w, b):
    X = np.asarray(X, dtype=float)
    return sigmoid(X @ w + b).reshape(-1)

def predict(X, w, b, mean, std):
    # Normalize with the TRAINING statistics, not this set's own.
    x = apply_normalizer(X, mean, std)

    # if y_hat >= 0.5 --> round up to 1
    # if y_hat < 0.5 --> round down to 0
    probabilities = predict_proba(x, w, b)
    return (probabilities >= 0.5).astype(int) # Threshold = 0.5


In [ ]:
# Main code
from sklearn.linear_model import LogisticRegression

# Train the custom model on 75% of the training set.
w, b, train_losses, mu, sigma = train(X_train, y_train, 500, 0.1)

# Train sklearn on the SAME normalised features. Comparing a normalised
# custom model against an unnormalised sklearn model would measure the
# preprocessing, not the implementation -- and the rubric asks specifically
# for "comparative performance compared to sklearn logistic regression".
X_train_n = apply_normalizer(X_train, mu, sigma)
X_validate_n = apply_normalizer(X_validate, mu, sigma)
sklearn_model = LogisticRegression(C=np.inf, max_iter=1000).fit(X_train_n, y_train)

# Predict on the held-out 25% for comparison.
custom_pred = predict(X_validate, w, b, mu, sigma)
sklearn_pred = sklearn_model.predict(X_validate_n)


In [ ]:
# Predict values in test dataset
# Retrain the custom model using the full train dataset (100%)
w, b, test_losses, mu_full, sigma_full = train(X, y, 500, 0.1)

# Predict the test set, normalising with the FULL-train statistics.
custom_pred_test = predict(X_test_submission, w, b, mu_full, sigma_full)

# Save predictions. The brief asks for this exact filename.
from pathlib import Path
out_dir = Path("outputs"); out_dir.mkdir(parents=True, exist_ok=True)
submission_df = pd.DataFrame({
    "id": test_features_df.iloc[:, 0].to_numpy(),
    "label": custom_pred_test,
})
submission_df.to_csv(out_dir / "LogReg_predictions.csv", index=False)
print(submission_df["label"].value_counts())


In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.show()

Comparison of Custom Model Against Sklearn Model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Custom accuracy:", accuracy_score(y_validate, custom_pred))
print("Sklearn accuracy:", accuracy_score(y_validate, sklearn_pred))

print("Custom precision:", precision_score(y_validate, custom_pred))
print("Sklearn precision:", precision_score(y_validate, sklearn_pred))

print("Custom recall:", recall_score(y_validate, custom_pred))
print("Sklearn recall:", recall_score(y_validate, sklearn_pred))

print("Custom F1:", f1_score(y_validate, custom_pred))
print("Sklearn F1:", f1_score(y_validate, sklearn_pred))

print("Custom confusion matrix:")
print(confusion_matrix(y_validate, custom_pred))

print("Sklearn confusion matrix:")
print(confusion_matrix(y_validate, sklearn_pred))